# KNNFewShot Benchmark (Chapter 6)

This notebook accompanies Chapter 6 §6.3 of *Context Engineering with DSPy*. KNNFewShot uses sentence embeddings to pick the *k* most-similar training examples for each input at inference time, then re-bootstraps a BootstrapFewShot program with those demos.

KNNFewShot doesn't play well with `dspy.Evaluate` (its dynamic forward replacement breaks the evaluator's introspection), so this notebook uses a manual evaluation loop. Requires `sentence-transformers`.


## Setup

In [ ]:
%pip install -r ../requirements.txt -q sentence-transformers

In [ ]:
import dspy
import pandas as pd
import time
import os
import random
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any
from dotenv import load_dotenv

load_dotenv()

PROGRAMS_DIR = Path("optimized_programs")
PROGRAMS_DIR.mkdir(exist_ok=True)

NUM_THREADS = 16

# Pricing per 1M tokens (as of Dec 2025)
PRICING = {
    "openai/gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "openai/gpt-4o": {"input": 2.50, "output": 10.00},
}


def calculate_cost(usage: dict) -> float:
    total_cost = 0.0
    for model, data in usage.items():
        if model in PRICING:
            prompt_tokens = data.get("prompt_tokens", 0)
            completion_tokens = data.get("completion_tokens", 0)
            total_cost += (prompt_tokens / 1_000_000) * PRICING[model]["input"]
            total_cost += (completion_tokens / 1_000_000) * PRICING[model]["output"]
    return total_cost


@dataclass
class BenchmarkResult:
    optimizer_name: str
    baseline_accuracy: float
    optimized_accuracy: float
    accuracy_uplift: float
    total_tokens: int
    prompt_tokens: int
    completion_tokens: int
    cost_usd: float
    time_seconds: float
    notes: str = ""
    usage_by_model: dict = field(default_factory=dict)
    program: Any = None


def print_benchmark_result(result: BenchmarkResult):
    print("\n" + "="*60)
    print(f"OPTIMIZER: {result.optimizer_name}")
    print("="*60)
    print(f"Baseline Accuracy:   {result.baseline_accuracy*100:.1f}%")
    print(f"Optimized Accuracy:  {result.optimized_accuracy*100:.1f}%")
    print(f"Accuracy Uplift:     {result.accuracy_uplift*100:+.1f}%")
    print("-"*60)
    print(f"Total Tokens:        {result.total_tokens:,}")
    print(f"  - Prompt:          {result.prompt_tokens:,}")
    print(f"  - Completion:      {result.completion_tokens:,}")
    print(f"Estimated Cost:      ${result.cost_usd:.4f}")
    print(f"Time Taken:          {result.time_seconds:.1f}s")
    if result.notes:
        print(f"Notes:               {result.notes}")
    print("="*60 + "\n")


def save_program(program, name: str):
    json_path = PROGRAMS_DIR / f"{name}.json"
    program_info = {"name": name, "predictors": []}
    for pred_name, predictor in program.named_predictors():
        pred_info = {
            "name": pred_name,
            "signature": str(predictor.signature) if hasattr(predictor, 'signature') else None,
            "instructions": predictor.signature.instructions if hasattr(predictor, 'signature') and hasattr(predictor.signature, 'instructions') else None,
            "demos": [],
        }
        if hasattr(predictor, 'demos') and predictor.demos:
            for demo in predictor.demos:
                demo_dict = {}
                if isinstance(demo, dict):
                    demo_dict = {k: str(v) if not isinstance(v, (str, int, float, bool, type(None))) else v for k, v in demo.items()}
                elif hasattr(demo, 'toDict'):
                    demo_dict = {k: str(v) if not isinstance(v, (str, int, float, bool, type(None))) else v for k, v in demo.toDict().items()}
                elif hasattr(demo, '__dict__'):
                    demo_dict = {k: str(v) if not isinstance(v, (str, int, float, bool, type(None))) else v for k, v in demo.__dict__.items()}
                pred_info["demos"].append(demo_dict)
        program_info["predictors"].append(pred_info)
    with open(json_path, 'w') as f:
        json.dump(program_info, f, indent=2, default=str)
    # DSPy 3.x requires the path to end with .json or .pkl; .dspy is not supported.
    state_path = PROGRAMS_DIR / f"{name}_state.json"
    try:
        program.save(str(state_path))
        print(f"Saved program to {json_path} and {state_path}")
    except Exception as e:
        print(f"Saved program to {json_path} (DSPy state save failed: {e})")
    return json_path


class DetectAIText(dspy.Signature):
    """Detects whether text is AI-generated."""
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="Whether the text is AI-generated")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def exact_match(example, response, trace=None):
    return 1 if example.is_ai == response.is_ai else 0

## Run optimizer

In [ ]:
# Check for sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    print("sentence-transformers is available")
except ImportError:
    raise SystemExit("ERROR: sentence-transformers not installed. Install with: pip install sentence-transformers")

print("Loading sentence transformer model...")
st_model = SentenceTransformer("all-MiniLM-L6-v2")
vectorizer = dspy.Embedder(st_model.encode)

print("\nInitializing LM...")
task_lm = dspy.LM(
    "openai/gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=16000,
    cache=False,
)
dspy.configure(lm=task_lm)

print("\nLoading dataset...")
csv_path = '../data/ai_vs_human200.csv'
df = pd.read_csv(csv_path)
examples = df.to_dict(orient='records')
dataset = [dspy.Example(**ex).with_inputs("text") for ex in examples]

random.seed(42)
random.shuffle(dataset)
n = len(dataset)
train_end = int(n * 0.5)
val_end = int(n * 0.75)
trainset = dataset[:train_end]
valset = dataset[train_end:val_end]
testset = dataset[val_end:]

print(f"Training set:   {len(trainset)} examples")
print(f"Validation set: {len(valset)} examples")
print(f"Test set:       {len(testset)} examples")

print("\nRunning baseline evaluation on test set...")
detector = AIDetector()
evaluator = dspy.Evaluate(
    devset=testset, metric=exact_match,
    num_threads=NUM_THREADS, display_progress=True, display_table=False,
)
baseline_result = evaluator(detector)
baseline_accuracy = baseline_result.score / 100
print(f"Baseline accuracy: {baseline_accuracy*100:.1f}%")

print("\n" + "="*60)
print("Running KNNFewShot optimizer...")
print("="*60)

from dspy.teleprompt import KNNFewShot

start_time = time.time()
with dspy.track_usage() as usage:
    optimizer = KNNFewShot(
        k=4,
        trainset=trainset,
        vectorizer=vectorizer,
    )
    program_knnfewshot = optimizer.compile(AIDetector())

    # KNNFewShot doesn't play nice with dspy.Evaluate (dynamic forward replacement),
    # so we evaluate with a manual loop.
    print("\nEvaluating optimized program on test set...")
    correct = 0
    total = 0
    for i, ex in enumerate(testset):
        try:
            result = program_knnfewshot(text=ex.text)
            if ex.is_ai == result.is_ai:
                correct += 1
            total += 1
            if (i + 1) % 10 == 0:
                print(f"  Progress: {i+1}/{len(testset)}, Accuracy: {correct}/{total} ({100*correct/total:.1f}%)")
        except Exception as e:
            print(f"  Error on example {i}: {e}")
            total += 1

    optimized_accuracy = correct / total if total > 0 else 0
    print(f"\nFinal accuracy: {correct}/{total} ({100*optimized_accuracy:.1f}%)")

elapsed = time.time() - start_time
total_usage = usage.get_total_tokens()

total_tokens = sum(d.get("total_tokens", 0) for d in total_usage.values())
prompt_tokens = sum(d.get("prompt_tokens", 0) for d in total_usage.values())
completion_tokens = sum(d.get("completion_tokens", 0) for d in total_usage.values())

result = BenchmarkResult(
    optimizer_name="KNNFewShot",
    baseline_accuracy=baseline_accuracy,
    optimized_accuracy=optimized_accuracy,
    accuracy_uplift=optimized_accuracy - baseline_accuracy,
    total_tokens=total_tokens,
    prompt_tokens=prompt_tokens,
    completion_tokens=completion_tokens,
    cost_usd=calculate_cost(total_usage),
    time_seconds=elapsed,
    notes="Dynamic KNN demo selection per query. Re-bootstraps on each call. k=4 demos.",
    usage_by_model=total_usage,
    program=program_knnfewshot,
)


## Results

In [ ]:
print_benchmark_result(result)
save_program(program_knnfewshot, "knnfewshot")

print("\n" + "="*60)
print("PROGRAM DETAILS: KNNFewShot")
print("="*60)

for pred_name, predictor in program_knnfewshot.named_predictors():
    print(f"\nPredictor: {pred_name}")
    print("-"*40)
    if hasattr(predictor, 'signature') and hasattr(predictor.signature, 'instructions'):
        instr = predictor.signature.instructions
        if instr:
            print(f"Instructions: {instr[:200]}..." if len(instr) > 200 else f"Instructions: {instr}")
    if hasattr(predictor, 'demos') and predictor.demos:
        print(f"Number of demos: {len(predictor.demos)}")
    else:
        print("Note: KNNFewShot selects demos dynamically at inference time")

results_df = pd.DataFrame([{
    "optimizer": result.optimizer_name,
    "baseline_accuracy": result.baseline_accuracy,
    "test_accuracy": result.optimized_accuracy,
    "accuracy_uplift": result.accuracy_uplift,
    "total_tokens": result.total_tokens,
    "prompt_tokens": result.prompt_tokens,
    "completion_tokens": result.completion_tokens,
    "cost_usd": result.cost_usd,
    "time_seconds": result.time_seconds,
    "notes": result.notes,
}])

results_df.to_csv("knnfewshot_benchmark_results.csv", index=False)
print("\nResults saved to knnfewshot_benchmark_results.csv")
